# Dinosaur sparsification example

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import trimesh

from mesh_to_housing import MeshProcessor
from mesh_to_housing.geometry.simplification import simplify_mesh

pio.renderers.default = "plotly_mimetype+notebook"
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "mesh-assets/dinosaur.dae").is_file())
mesh_path = root / "mesh-assets/dinosaur.dae"


In [20]:
mesh = trimesh.load_scene(mesh_path, process=False).to_mesh()
print(f"{mesh_path.name}: {len(mesh.vertices):,} vertices, {len(mesh.faces):,} triangles")


def mesh_trace(model, opacity=1.0):
    x, y, z = model.vertices.T
    i, j, k = model.faces.T
    return go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color="#a8b5c4",
                     opacity=opacity, name="Mesh", hoverinfo="skip")


def point_trace(result):
    x, y, z = result.points.T
    colors = np.where(result.important_mask, "#d18a00", "#d94545")
    return go.Scatter3d(x=x, y=y, z=z, mode="markers", name="Selected points",
                        marker=dict(size=3, color=colors, opacity=1))


def show(*traces, title):
    fig = go.Figure(data=list(traces))
    fig.update_layout(title=title, template="plotly_white", height=650,
                      scene=dict(aspectmode="data", camera=dict(up=dict(x=0, y=0, z=1))),
                      margin=dict(l=0, r=0, t=45, b=0))
    fig.show()

show(mesh_trace(mesh), title="Original mesh")


dinosaur.dae: 11,677 vertices, 22,294 triangles


## Point selection

Select representative points from a mesh using one of three methods: `random`, `normal_variation`, or `farthest`.

- `random` samples mesh vertices uniformly.
- `normal_variation` gives higher priority to vertices where neighboring face normals differ more strongly. This tends to preserve sharp edges, creases, corners, and open boundaries. Set `strength=0` to remove this preference.
- `farthest` spreads selected points across the mesh by repeatedly choosing points that are far from those already selected.

All three methods currently select from the mesh's existing vertices rather than sampling the surface continuously. As a result, areas with denser triangulation contain more candidate points and can influence the final selection.

`n_points` specifies the total number of output points.


In [ ]:
n_points = 300
seed = 42
important_points = np.empty((0, 3))

In [22]:
method = "random"

processor = MeshProcessor(mesh)
result = processor.run(n_points, method=method, seed=seed,
                       important_points=important_points)

show(mesh_trace(mesh, opacity=0.35), point_trace(result),
     title=f"{method}: {n_points} points over the mesh")

In [21]:
method = "normal_variation"
strength = 8.0

processor = MeshProcessor(mesh)
result = processor.run(n_points, method=method, seed=seed,
                       important_points=important_points, strength = strength)

show(mesh_trace(mesh, opacity=0.35), point_trace(result),
     title=f"{method}: {n_points} points over the mesh")

In [24]:
method = "farthest"

processor = MeshProcessor(mesh)
result = processor.run(n_points, method=method, seed=seed,
                       important_points=important_points)

show(mesh_trace(mesh, opacity=0.35), point_trace(result),
     title=f"{method}: {n_points} points over the mesh")


## Mesh simplification

A separate quadric-decimation preview of the original mesh. It has an approximate triangle
budget, does not preserve anchors, and is not a triangulation of the selected points.


In [9]:
target_faces = 300
simplified = simplify_mesh(processor.mesh, face_count=target_faces)
print(f"{len(simplified.vertices):,} vertices, {len(simplified.faces):,} triangles")
show(mesh_trace(simplified), title="Quadric-decimated mesh")


152 vertices, 300 triangles


In [25]:
x, y, z = simplified.vertices.T
simplified_vertices = go.Scatter3d(
    x=x, y=y, z=z,
    mode="markers",
    marker=dict(size=3, color="#d94545"),
    name="Mesh vertices",
)

print(f"{len(simplified.vertices)} vertices")
show(
    mesh_trace(mesh, opacity=0.35),
    simplified_vertices,
    title="Simplified mesh with vertices",
)

152 vertices
